# Phase 5 — Model Migration: Credit Risk FPD (OCI Data Science)

**Hackathon PoD Academy** — Claro + Oracle

| Item | Valor |
|------|-------|
| **Pipeline** | Fabric → OCI Migration |
| **Modelos** | Logistic Regression L1 (Scorecard) + LightGBM (GBDT) |
| **Features** | 59 selecionadas (de 398 iniciais) |
| **Target** | FPD (First Payment Default) |
| **Dados** | 3.9M registros, 6 SAFRAs (202410-202503) |
| **Shape** | VM.Standard.E4.Flex — 24 OCPUs, 384 GB RAM |
| **Quality Gate** | QG-05: KS>20, AUC>0.65, Gini>30, PSI<0.25 |

---

## 1. Setup — Threading, Imports & Compatibility Patch

In [ ]:
import os, time, json, pickle, uuid
from datetime import datetime

# ── Threading (BEFORE numpy/sklearn) ──
NCPUS = int(os.environ.get("NOTEBOOK_OCPUS", "24"))
os.environ["OMP_NUM_THREADS"] = str(NCPUS)
os.environ["OPENBLAS_NUM_THREADS"] = str(NCPUS)
os.environ["MKL_NUM_THREADS"] = "1"
print(f"Threading: OMP={NCPUS}, OPENBLAS={NCPUS}, MKL=1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# ── sklearn/LightGBM compat patch ──
import sklearn.utils.validation as _val
_orig = _val.check_array
def _patch(*a, **kw):
    kw.pop("force_all_finite", None)
    kw.pop("ensure_all_finite", None)
    return _orig(*a, **kw)
_val.check_array = _patch

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve
from scipy.stats import ks_2samp
import lightgbm as lgb
from category_encoders import CountEncoder

# Plot style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 100

# OCI Auth
try:
    import ads
    ads.set_auth("resource_principal")
    print("OCI Auth: Resource Principal")
except Exception:
    print("WARN: ADS not available — local mode")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
ARTIFACT_DIR = "/home/datascience/artifacts"
os.makedirs(f"{ARTIFACT_DIR}/models", exist_ok=True)
os.makedirs(f"{ARTIFACT_DIR}/metrics", exist_ok=True)
os.makedirs(f"{ARTIFACT_DIR}/plots", exist_ok=True)
print(f"Run ID: {RUN_ID} | sklearn {sklearn.__version__} | lgbm {lgb.__version__}")

## 2. Configuration — Features, Split, Baseline

In [ ]:
# ── Object Storage config ──
NAMESPACE = os.environ.get("OCI_NAMESPACE", "SET_YOUR_NAMESPACE")
GOLD_BUCKET = os.environ.get("GOLD_BUCKET", "pod-academy-gold")
GOLD_PATH = f"oci://{GOLD_BUCKET}@{NAMESPACE}/clientes_consolidado/"
LOCAL_DATA_PATH = "/home/datascience/data/clientes_consolidado/"

TARGET = "FPD"

# ── 59 selected features (exact Fabric v6) ──
SELECTED_FEATURES = [
    "TARGET_SCORE_02", "TARGET_SCORE_01",
    "REC_SCORE_RISCO", "REC_TAXA_STATUS_A", "REC_QTD_LINHAS",
    "REC_DIAS_ENTRE_RECARGAS", "REC_QTD_INST_DIST_REG",
    "REC_DIAS_DESDE_ULTIMA_RECARGA", "REC_TAXA_CARTAO_ONLINE",
    "REC_QTD_STATUS_ZB2", "REC_QTD_CARTAO_ONLINE",
    "REC_COEF_VARIACAO_REAL", "var_26",
    "FAT_DIAS_MEDIO_CRIACAO_VENCIMENTO", "REC_VLR_CREDITO_STDDEV",
    "REC_TAXA_PLAT_PREPG", "REC_VLR_REAL_STDDEV",
    "REC_QTD_CARTAO_CHIPPRE", "REC_QTD_PLANOS", "REC_QTD_PLAT_AUTOC",
    "PAG_QTD_PAGAMENTOS_TOTAL", "FAT_QTD_FATURAS_PRIMEIRA", "var_73",
    "REC_QTD_STATUS_ZB1", "FAT_TAXA_PRIMEIRA_FAT", "FAT_DIAS_ATRASO_MIN",
    "PAG_TAXA_PAGAMENTOS_COM_JUROS", "FAT_DIAS_MAX_CRIACAO_VENCIMENTO",
    "REC_COEF_VARIACAO_CREDITO", "PAG_DIAS_ENTRE_FATURAS",
    "FAT_DIAS_DESDE_ATIVACAO_CONTA", "var_85", "REC_FREQ_RECARGA_DIARIA",
    "FAT_DIAS_DESDE_ULTIMA_FAT", "FAT_DIAS_DESDE_PRIMEIRA_FAT",
    "PAG_QTD_FATURAS_DISTINTAS", "var_82", "PAG_DIAS_DESDE_ULTIMA_FATURA",
    "REC_TAXA_PLAT_CONTROLE", "var_90", "PAG_TAXA_FORMA_PA",
    "PAG_QTD_PAGAMENTOS_COM_JUROS", "PAG_QTD_AREAS",
    "FAT_DIAS_ATRASO_MEDIO", "REC_QTD_RECARGAS_TOTAL", "var_28", "var_44",
    "PAG_VLR_PAGAMENTO_FATURA_STDDEV", "REC_QTD_TIPOS_RECARGA",
    "var_34", "FAT_QTD_FAT_PREPG", "PAG_FLAG_ALTO_RISCO", "var_67",
    "REC_QTD_PLATAFORMAS", "PAG_QTD_STATUS_R",
    "PAG_COEF_VARIACAO_PAGAMENTO", "FAT_QTD_FATURAS_ACA",
    "REC_QTD_INSTITUICOES", "FAT_QTD_SAFRAS_DISTINTAS",
]

CAT_FEATURES = ["var_34", "var_67"]
NUM_FEATURES = [f for f in SELECTED_FEATURES if f not in CAT_FEATURES]

TRAIN_SAFRAS = [202410, 202411, 202412, 202501]
OOS_SAFRA = [202501]
OOT_SAFRAS = [202502, 202503]

FABRIC_BASELINE = {
    "lgbm": {"ks_oot": 0.33974, "auc_oot": 0.73032, "gini_oot": 46.064,
             "ks_oos": 0.34971, "auc_oos": 0.73805},
    "lr_l1": {"ks_oot": 0.32767, "auc_oot": 0.72073, "gini_oot": 44.146,
              "ks_oos": 0.33846, "auc_oos": 0.72902},
}

print(f"Features: {len(SELECTED_FEATURES)} ({len(NUM_FEATURES)} num + {len(CAT_FEATURES)} cat)")
print(f"Train SAFRAs: {TRAIN_SAFRAS}")
print(f"OOT SAFRAs: {OOT_SAFRAS}")

## 3. Data Loading

In [ ]:
import pyarrow.parquet as pq

columns_to_load = SELECTED_FEATURES + [TARGET, "SAFRA", "NUM_CPF"]
t0 = time.time()

if os.path.exists(LOCAL_DATA_PATH):
    print(f"Reading from local block volume: {LOCAL_DATA_PATH}")
    table = pq.read_table(LOCAL_DATA_PATH, columns=columns_to_load)
    df = table.to_pandas(self_destruct=True)
else:
    print(f"Reading from Object Storage: {GOLD_PATH}")
    df = pd.read_parquet(GOLD_PATH, columns=columns_to_load)

elapsed = time.time() - t0
print(f"\nLoaded: {len(df):,} rows x {df.shape[1]} columns in {elapsed:.1f}s")
print(f"SAFRAs: {sorted(df['SAFRA'].unique())}")
print(f"FPD rate: {df[TARGET].mean():.4f} ({df[TARGET].sum():,} defaults / {len(df):,} total)")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

## 4. Temporal Split

In [ ]:
df_train = df[df["SAFRA"].isin(TRAIN_SAFRAS)].copy()
df_oos = df[df["SAFRA"].isin(OOS_SAFRA)].copy()
df_oot = df[df["SAFRA"].isin(OOT_SAFRAS)].copy()

X_train, y_train = df_train[SELECTED_FEATURES], df_train[TARGET]
X_oos, y_oos = df_oos[SELECTED_FEATURES], df_oos[TARGET]
X_oot, y_oot = df_oot[SELECTED_FEATURES], df_oot[TARGET]

# Validate no leakage
assert set(df_train["SAFRA"].unique()).isdisjoint(set(df_oot["SAFRA"].unique())), "SAFRA leak!"

split_summary = pd.DataFrame({
    "Split": ["Train", "OOS (202501)", "OOT (202502-03)"],
    "SAFRAs": [str(TRAIN_SAFRAS), str(OOS_SAFRA), str(OOT_SAFRAS)],
    "Records": [f"{len(df_train):,}", f"{len(df_oos):,}", f"{len(df_oot):,}"],
    "FPD Rate": [f"{df_train[TARGET].mean():.4f}", f"{df_oos[TARGET].mean():.4f}", f"{df_oot[TARGET].mean():.4f}"],
    "Defaults": [f"{df_train[TARGET].sum():,}", f"{df_oos[TARGET].sum():,}", f"{df_oot[TARGET].sum():,}"],
})
print(split_summary.to_string(index=False))
del df  # free memory

## 5. Model Training

### 5.1 LR L1 Scorecard (Interpretable Baseline)

In [ ]:
lr_pipeline = Pipeline(steps=[
    ("prep", ColumnTransformer(transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), NUM_FEATURES),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", CountEncoder(combine_min_nan_groups=True, normalize=True,
                                     handle_missing=0, handle_unknown=0)),
        ]), CAT_FEATURES),
    ])),
    ("model", LogisticRegression(
        C=0.5, penalty="l1", solver="liblinear",
        max_iter=2000, tol=0.001, class_weight="balanced", random_state=42,
    )),
])

t0 = time.time()
lr_pipeline.fit(X_train, y_train)
lr_time = time.time() - t0
print(f"LR L1 trained in {lr_time:.1f}s")

### 5.2 LightGBM GBDT (Maximum Discrimination)

In [ ]:
lgbm_pipeline = Pipeline(steps=[
    ("prep", ColumnTransformer(transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]), NUM_FEATURES),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", CountEncoder(combine_min_nan_groups=True, normalize=True,
                                     handle_missing=0, handle_unknown=0)),
        ]), CAT_FEATURES),
    ])),
    ("model", lgb.LGBMClassifier(
        objective="binary", n_estimators=250, learning_rate=0.05,
        max_depth=4, colsample_bytree=0.8,
        n_jobs=NCPUS, num_threads=NCPUS,
        random_state=42, verbosity=-1,
    )),
])

t0 = time.time()
lgbm_pipeline.fit(X_train, y_train)
lgbm_time = time.time() - t0
print(f"LightGBM trained in {lgbm_time:.1f}s (num_threads={NCPUS})")

## 6. Metrics — KS, AUC, Gini

In [ ]:
def compute_ks(y_true, y_prob):
    ks_stat, _ = ks_2samp(y_prob[y_true == 0], y_prob[y_true == 1])
    return ks_stat

def compute_gini(auc):
    return (2 * auc - 1) * 100

def eval_model(pipeline, X, y, label):
    prob = pipeline.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, prob)
    ks = compute_ks(y.values, prob)
    return {"KS": f"{ks:.5f}", "AUC": f"{auc:.5f}", "Gini": f"{compute_gini(auc):.2f}"}

# Compute all metrics
results = []
for name, pipe in [("LR L1", lr_pipeline), ("LightGBM", lgbm_pipeline)]:
    for split, X_s, y_s in [("Train", X_train, y_train), ("OOS", X_oos, y_oos), ("OOT", X_oot, y_oot)]:
        m = eval_model(pipe, X_s, y_s, split)
        results.append({"Model": name, "Split": split, **m})

metrics_df = pd.DataFrame(results)
print("\n" + "=" * 65)
print("MODEL METRICS")
print("=" * 65)
print(metrics_df.to_string(index=False))

## 7. Visualization Panel 1 — Performance Curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("Phase 5 — Model Performance (OCI Data Science)", fontsize=16, fontweight="bold")

colors = {"LR L1": "#2196F3", "LightGBM": "#FF5722"}
splits_data = {"OOS": (X_oos, y_oos), "OOT": (X_oot, y_oot)}

# ── ROC Curves ──
for idx, (split_name, (X_s, y_s)) in enumerate(splits_data.items()):
    ax = axes[0, idx]
    for name, pipe in [("LR L1", lr_pipeline), ("LightGBM", lgbm_pipeline)]:
        prob = pipe.predict_proba(X_s)[:, 1]
        fpr, tpr, _ = roc_curve(y_s, prob)
        auc_val = roc_auc_score(y_s, prob)
        ax.plot(fpr, tpr, color=colors[name], label=f"{name} (AUC={auc_val:.4f})", linewidth=2)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax.set_title(f"ROC Curve — {split_name}", fontweight="bold")
    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR")
    ax.legend(loc="lower right")

# ── KS Curves ──
ax = axes[0, 2]
for name, pipe in [("LR L1", lr_pipeline), ("LightGBM", lgbm_pipeline)]:
    prob = pipe.predict_proba(X_oot)[:, 1]
    thresholds = np.linspace(0, 1, 100)
    good_cdf = np.array([np.mean(prob[y_oot == 0] <= t) for t in thresholds])
    bad_cdf = np.array([np.mean(prob[y_oot == 1] <= t) for t in thresholds])
    ks_vals = np.abs(good_cdf - bad_cdf)
    ks_max_idx = np.argmax(ks_vals)
    ax.plot(thresholds, ks_vals, color=colors[name], label=f"{name} (KS={ks_vals[ks_max_idx]:.4f})", linewidth=2)
ax.axhline(y=0.20, color="red", linestyle="--", alpha=0.4, label="Threshold (0.20)")
ax.set_title("KS Curve — OOT", fontweight="bold")
ax.set_xlabel("Threshold")
ax.set_ylabel("KS")
ax.legend()

# ── Score Distribution ──
for idx, (name, pipe) in enumerate([("LR L1", lr_pipeline), ("LightGBM", lgbm_pipeline)]):
    ax = axes[1, idx]
    prob_oot = pipe.predict_proba(X_oot)[:, 1]
    scores = ((1 - prob_oot) * 1000).astype(int).clip(0, 1000)
    ax.hist(scores[y_oot == 0], bins=50, alpha=0.6, color="green", label="Non-default", density=True)
    ax.hist(scores[y_oot == 1], bins=50, alpha=0.6, color="red", label="Default (FPD)", density=True)
    ax.axvline(x=300, color="black", linestyle="--", alpha=0.3)
    ax.axvline(x=500, color="black", linestyle="--", alpha=0.3)
    ax.axvline(x=700, color="black", linestyle="--", alpha=0.3)
    ax.set_title(f"Score Distribution — {name} OOT", fontweight="bold")
    ax.set_xlabel("Score (0-1000)")
    ax.legend()

# ── Fabric Parity Comparison ──
ax = axes[1, 2]
metrics_compare = []
for name, pipe, baseline_key in [("LR L1", lr_pipeline, "lr_l1"), ("LightGBM", lgbm_pipeline, "lgbm")]:
    prob_oot = pipe.predict_proba(X_oot)[:, 1]
    oci_ks = compute_ks(y_oot.values, prob_oot)
    oci_auc = roc_auc_score(y_oot, prob_oot)
    fab_ks = FABRIC_BASELINE[baseline_key]["ks_oot"]
    fab_auc = FABRIC_BASELINE[baseline_key]["auc_oot"]
    metrics_compare.append({"Model": name, "Metric": "KS", "Fabric": fab_ks, "OCI": oci_ks})
    metrics_compare.append({"Model": name, "Metric": "AUC", "Fabric": fab_auc, "OCI": oci_auc})

comp_df = pd.DataFrame(metrics_compare)
x_pos = np.arange(len(comp_df))
width = 0.35
ax.bar(x_pos - width/2, comp_df["Fabric"], width, label="Fabric", color="#607D8B", alpha=0.8)
ax.bar(x_pos + width/2, comp_df["OCI"], width, label="OCI", color="#FF9800", alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels([f"{r['Model']}\n{r['Metric']}" for _, r in comp_df.iterrows()], fontsize=9)
ax.set_title("Fabric vs OCI Parity — OOT", fontweight="bold")
ax.legend()

plt.tight_layout()
plt.savefig(f"{ARTIFACT_DIR}/plots/panel1_performance_{RUN_ID}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACT_DIR}/plots/panel1_performance_{RUN_ID}.png")

## 8. Decile Analysis

In [ ]:
def decile_table(y_true, y_prob, label=""):
    df_e = pd.DataFrame({"target": y_true.values, "prob": y_prob})
    df_e["decile"] = pd.qcut(df_e["prob"], q=10, labels=False, duplicates="drop") + 1
    t = df_e.groupby("decile").agg(
        count=("target", "count"), events=("target", "sum"),
        avg_prob=("prob", "mean"), min_prob=("prob", "min"), max_prob=("prob", "max"),
    ).reset_index()
    t["event_rate"] = t["events"] / t["count"]
    t["cum_events"] = t["events"].cumsum()
    t["cum_event_pct"] = t["cum_events"] / t["events"].sum()
    t["lift"] = t["event_rate"] / (t["events"].sum() / t["count"].sum())
    return t

# LGBM OOT decile
lgbm_prob_oot = lgbm_pipeline.predict_proba(X_oot)[:, 1]
decile_oot = decile_table(y_oot, lgbm_prob_oot, "LGBM OOT")

print("\nLightGBM — Decile Table (OOT)")
print("=" * 85)
print(decile_oot[["decile", "count", "events", "event_rate", "cum_event_pct", "lift"]]
      .to_string(index=False, float_format="%.4f"))
print(f"\nMonotonic: {decile_oot['event_rate'].is_monotonic_increasing}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax1, ax2 = axes

ax1.bar(decile_oot["decile"], decile_oot["event_rate"] * 100, color="#FF5722", alpha=0.8)
ax1.axhline(y=decile_oot["events"].sum()/decile_oot["count"].sum()*100, color="black", linestyle="--", alpha=0.4, label="Avg FPD rate")
ax1.set_xlabel("Decile")
ax1.set_ylabel("FPD Rate (%)")
ax1.set_title("LightGBM — Default Rate by Decile (OOT)", fontweight="bold")
ax1.legend()

ax2.bar(decile_oot["decile"], decile_oot["lift"], color="#2196F3", alpha=0.8)
ax2.axhline(y=1.0, color="black", linestyle="--", alpha=0.4, label="Baseline (lift=1)")
ax2.set_xlabel("Decile")
ax2.set_ylabel("Lift")
ax2.set_title("LightGBM — Lift by Decile (OOT)", fontweight="bold")
ax2.legend()

plt.tight_layout()
plt.savefig(f"{ARTIFACT_DIR}/plots/decile_analysis_{RUN_ID}.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Feature Importance (LightGBM Top 20)

In [ ]:
lgbm_model = lgbm_pipeline.named_steps["model"]
feat_names = [f"num__{f}" for f in NUM_FEATURES] + [f"cat__{f}" for f in CAT_FEATURES]
importance = pd.DataFrame({"feature": feat_names, "importance": lgbm_model.feature_importances_})
importance = importance.sort_values("importance", ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(importance)), importance["importance"].values, color="#FF5722", alpha=0.8)
ax.set_yticks(range(len(importance)))
ax.set_yticklabels(importance["feature"].values)
ax.invert_yaxis()
ax.set_xlabel("Importance (gain)")
ax.set_title("LightGBM — Top 20 Feature Importance", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{ARTIFACT_DIR}/plots/feature_importance_{RUN_ID}.png", dpi=150, bbox_inches="tight")
plt.show()

print(importance.to_string(index=False))

## 10. PSI — Population Stability Index

In [ ]:
def compute_psi(expected, actual, bins=10):
    bp = np.percentile(expected, np.linspace(0, 100, bins + 1))
    bp = np.unique(bp); bp[0] = -np.inf; bp[-1] = np.inf
    e_cnt = np.histogram(expected, bins=bp)[0]
    a_cnt = np.histogram(actual, bins=bp)[0]
    e_pct = (e_cnt + 1) / (len(expected) + len(bp) - 1)
    a_pct = (a_cnt + 1) / (len(actual) + len(bp) - 1)
    return np.sum((a_pct - e_pct) * np.log(a_pct / e_pct))

# Score PSI
lr_train_s = lr_pipeline.predict_proba(X_train)[:, 1]
lr_oot_s = lr_pipeline.predict_proba(X_oot)[:, 1]
lgbm_train_s = lgbm_pipeline.predict_proba(X_train)[:, 1]
lgbm_oot_s = lgbm_pipeline.predict_proba(X_oot)[:, 1]

lr_psi = compute_psi(lr_train_s, lr_oot_s)
lgbm_psi = compute_psi(lgbm_train_s, lgbm_oot_s)

def psi_alert(psi):
    if psi < 0.10: return "GREEN (Stable)"
    elif psi < 0.25: return "YELLOW (Moderate shift)"
    else: return "RED (Significant shift!)"

print(f"Score PSI:")
print(f"  LR L1:     {lr_psi:.6f} — {psi_alert(lr_psi)}")
print(f"  LightGBM:  {lgbm_psi:.6f} — {psi_alert(lgbm_psi)}")

## 11. Fabric Parity Comparison

In [ ]:
# Compute OCI metrics for parity comparison
lr_prob_oos = lr_pipeline.predict_proba(X_oos)[:, 1]
lr_prob_oot = lr_pipeline.predict_proba(X_oot)[:, 1]
lgbm_prob_oos = lgbm_pipeline.predict_proba(X_oos)[:, 1]

oci_metrics = {
    "lr_l1": {
        "ks_oot": compute_ks(y_oot.values, lr_prob_oot),
        "auc_oot": roc_auc_score(y_oot, lr_prob_oot),
        "gini_oot": compute_gini(roc_auc_score(y_oot, lr_prob_oot)),
        "ks_oos": compute_ks(y_oos.values, lr_prob_oos),
        "auc_oos": roc_auc_score(y_oos, lr_prob_oos),
    },
    "lgbm": {
        "ks_oot": compute_ks(y_oot.values, lgbm_prob_oot),
        "auc_oot": roc_auc_score(y_oot, lgbm_prob_oot),
        "gini_oot": compute_gini(roc_auc_score(y_oot, lgbm_prob_oot)),
        "ks_oos": compute_ks(y_oos.values, lgbm_prob_oos),
        "auc_oos": roc_auc_score(y_oos, lgbm_prob_oos),
    },
}

print("\n" + "=" * 80)
print("PARITY REPORT: OCI vs Fabric")
print("=" * 80)
parity_rows = []
for model_key in ["lr_l1", "lgbm"]:
    model_label = "LR L1" if model_key == "lr_l1" else "LightGBM"
    for metric_key in FABRIC_BASELINE[model_key]:
        fab = FABRIC_BASELINE[model_key][metric_key]
        oci = oci_metrics[model_key][metric_key]
        diff = abs(oci - fab)
        tol = 0.02 if "ks" in metric_key else (0.005 if "auc" in metric_key else 2.0)
        status = "PASS" if diff <= tol else "FAIL"
        parity_rows.append({
            "Model": model_label, "Metric": metric_key,
            "Fabric": f"{fab:.5f}", "OCI": f"{oci:.5f}",
            "Diff": f"{diff:.5f}", "Status": status,
        })

parity_df = pd.DataFrame(parity_rows)
print(parity_df.to_string(index=False))
n_pass = (parity_df["Status"] == "PASS").sum()
n_total = len(parity_df)
print(f"\nParity: {n_pass}/{n_total} checks passed")

## 12. Quality Gate QG-05

In [ ]:
gates = [
    ("KS OOT > 0.20 (LR)",    oci_metrics["lr_l1"]["ks_oot"] > 0.20),
    ("KS OOT > 0.20 (LGBM)",  oci_metrics["lgbm"]["ks_oot"] > 0.20),
    ("AUC OOT > 0.65 (LR)",   oci_metrics["lr_l1"]["auc_oot"] > 0.65),
    ("AUC OOT > 0.65 (LGBM)", oci_metrics["lgbm"]["auc_oot"] > 0.65),
    ("Gini OOT > 30 (LR)",    oci_metrics["lr_l1"]["gini_oot"] > 30),
    ("Gini OOT > 30 (LGBM)",  oci_metrics["lgbm"]["gini_oot"] > 30),
    ("PSI < 0.25 (LR)",       lr_psi < 0.25),
    ("PSI < 0.25 (LGBM)",     lgbm_psi < 0.25),
    ("Decile monotonic",       decile_oot["event_rate"].is_monotonic_increasing),
]

print("\n" + "=" * 60)
print("QUALITY GATE QG-05 — Pre-Deployment Validation")
print("=" * 60)
all_passed = True
for name, passed in gates:
    s = "PASS" if passed else "FAIL"
    if not passed: all_passed = False
    print(f"  [{s}] {name}")

print(f"\n  Quality Gate QG-05: {'PASSED' if all_passed else 'FAILED'}")

## 13. Batch Scoring

In [ ]:
# Reload full data for scoring
if os.path.exists(LOCAL_DATA_PATH):
    table = pq.read_table(LOCAL_DATA_PATH, columns=columns_to_load)
    df_full = table.to_pandas(self_destruct=True)
else:
    df_full = pd.read_parquet(GOLD_PATH, columns=columns_to_load)

execution_id = str(uuid.uuid4())
prob_full = lgbm_pipeline.predict_proba(df_full[SELECTED_FEATURES])[:, 1]
scores_full = ((1 - prob_full) * 1000).astype(int).clip(0, 1000)

def risk_band(s):
    if s < 300: return "CRITICO"
    elif s < 500: return "ALTO"
    elif s < 700: return "MEDIO"
    else: return "BAIXO"

scores_df = pd.DataFrame({
    "NUM_CPF": df_full["NUM_CPF"].values,
    "SAFRA": df_full["SAFRA"].values,
    "SCORE": scores_full,
    "FAIXA_RISCO": [risk_band(s) for s in scores_full],
    "PROBABILIDADE_FPD": prob_full.round(6),
    "MODELO_VERSAO": "lgbm-oci-v1",
    "DATA_SCORING": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "EXECUTION_ID": execution_id,
})

print(f"Scored: {len(scores_df):,} records")
print(f"\nRisk Band Distribution:")
for band in ["CRITICO", "ALTO", "MEDIO", "BAIXO"]:
    count = (scores_df["FAIXA_RISCO"] == band).sum()
    pct = count / len(scores_df) * 100
    print(f"  {band:10s}: {count:>8,} ({pct:.1f}%)")

print(f"\nScore stats: mean={scores_full.mean():.0f}, median={int(np.median(scores_full))}, min={scores_full.min()}, max={scores_full.max()}")

In [ ]:
# Write scores to Gold bucket
SCORES_OUTPUT = f"oci://{GOLD_BUCKET}@{NAMESPACE}/clientes_scores/"

for safra in sorted(scores_df["SAFRA"].unique()):
    safra_df = scores_df[scores_df["SAFRA"] == safra]
    safra_path = f"{SCORES_OUTPUT}SAFRA={safra}/scores.parquet"
    safra_df.drop(columns=["SAFRA"]).to_parquet(safra_path, index=False)
    print(f"  SAFRA {safra}: {len(safra_df):,} records written")

# Local backup
os.makedirs(f"{ARTIFACT_DIR}/scoring", exist_ok=True)
scores_df.to_parquet(f"{ARTIFACT_DIR}/scoring/clientes_scores_all.parquet", index=False)
print(f"\nLocal backup saved: {ARTIFACT_DIR}/scoring/clientes_scores_all.parquet")

## 14. Model Registration (OCI Model Catalog)

In [ ]:
# Save models locally
with open(f"{ARTIFACT_DIR}/models/lr_l1_oci_{RUN_ID}.pkl", "wb") as f:
    pickle.dump(lr_pipeline, f)
with open(f"{ARTIFACT_DIR}/models/lgbm_oci_{RUN_ID}.pkl", "wb") as f:
    pickle.dump(lgbm_pipeline, f)

# Save feature list
with open(f"{ARTIFACT_DIR}/models/selected_features.json", "w") as f:
    json.dump(SELECTED_FEATURES, f, indent=2)

print(f"Models saved: {ARTIFACT_DIR}/models/")
print(f"  lr_l1_oci_{RUN_ID}.pkl")
print(f"  lgbm_oci_{RUN_ID}.pkl")
print(f"  selected_features.json")

# Register in OCI Model Catalog (requires ADS + Resource Principal)
try:
    from ads.model.generic_model import GenericModel
    from ads.common.model_metadata import UseCaseType
    
    for model_name, pipeline, metrics_dict in [
        ("credit-risk-fpd-lr-scorecard-v1", lr_pipeline, oci_metrics["lr_l1"]),
        ("credit-risk-fpd-lgbm-v1", lgbm_pipeline, oci_metrics["lgbm"]),
    ]:
        art_dir = f"{ARTIFACT_DIR}/catalog/{model_name}"
        os.makedirs(art_dir, exist_ok=True)
        with open(f"{art_dir}/model.pkl", "wb") as f:
            pickle.dump(pipeline, f)
        with open(f"{art_dir}/selected_features.json", "w") as f:
            json.dump(SELECTED_FEATURES, f)
        
        gm = GenericModel(estimator=pipeline, artifact_dir=art_dir)
        gm.prepare(inference_conda_env="generalml_p38_cpu_v1",
                   use_case_type=UseCaseType.BINARY_CLASSIFICATION,
                   force_overwrite=True)
        for k, v in metrics_dict.items():
            gm.metadata_custom.add(key=k, value=str(round(v, 5)), category="Training Profile")
        gm.metadata_custom.add(key="n_features", value="59", category="Training Profile")
        gm.metadata_custom.add(key="train_safras", value="202410-202501", category="Training Profile")
        
        model_id = gm.save(display_name=model_name,
                           description=f"Credit Risk FPD — {model_name}. 59 features, temporal split.")
        print(f"Registered: {model_name} → {model_id}")
        
except Exception as e:
    print(f"Model Catalog registration skipped: {e}")
    print("Models saved locally — register manually when ADS is available.")

## 15. Save Complete Results

In [ ]:
final_results = {
    "run_id": RUN_ID,
    "timestamp": datetime.now().isoformat(),
    "platform": "OCI Data Science",
    "shape": f"VM.Standard.E4.Flex ({NCPUS} OCPUs, 384 GB)",
    "training_time": {"lr_seconds": round(lr_time, 1), "lgbm_seconds": round(lgbm_time, 1)},
    "n_features": 59,
    "cat_features": CAT_FEATURES,
    "train_safras": TRAIN_SAFRAS,
    "oot_safras": OOT_SAFRAS,
    "oci_metrics": {k: {mk: round(mv, 5) for mk, mv in v.items()} for k, v in oci_metrics.items()},
    "fabric_baseline": FABRIC_BASELINE,
    "psi": {"lr": round(lr_psi, 6), "lgbm": round(lgbm_psi, 6)},
    "quality_gate_qg05": "PASSED" if all_passed else "FAILED",
    "records_scored": len(scores_df),
}

with open(f"{ARTIFACT_DIR}/metrics/training_results_{RUN_ID}.json", "w") as f:
    json.dump(final_results, f, indent=2, default=str)

print(json.dumps(final_results, indent=2, default=str))

## 16. Summary

### Execution Artifacts

| Artifact | Path |
|----------|------|
| LR L1 model | `artifacts/models/lr_l1_oci_{RUN_ID}.pkl` |
| LightGBM model | `artifacts/models/lgbm_oci_{RUN_ID}.pkl` |
| Feature list | `artifacts/models/selected_features.json` |
| Training metrics | `artifacts/metrics/training_results_{RUN_ID}.json` |
| Performance plots | `artifacts/plots/panel1_performance_{RUN_ID}.png` |
| Decile plots | `artifacts/plots/decile_analysis_{RUN_ID}.png` |
| Feature importance | `artifacts/plots/feature_importance_{RUN_ID}.png` |
| Batch scores | `artifacts/scoring/clientes_scores_all.parquet` |
| Gold scores | `oci://pod-academy-gold@ns/clientes_scores/` |

In [ ]:
print("=" * 70)
print(f"PHASE 5 COMPLETE — Run ID: {RUN_ID}")
print(f"Quality Gate QG-05: {final_results['quality_gate_qg05']}")
print(f"LR L1     KS OOT: {oci_metrics['lr_l1']['ks_oot']:.5f} | AUC: {oci_metrics['lr_l1']['auc_oot']:.5f}")
print(f"LightGBM  KS OOT: {oci_metrics['lgbm']['ks_oot']:.5f} | AUC: {oci_metrics['lgbm']['auc_oot']:.5f}")
print(f"Records scored: {len(scores_df):,}")
print(f"Training time: LR {lr_time:.1f}s + LGBM {lgbm_time:.1f}s")
print("=" * 70)
print("\nNext: Save this notebook with outputs for documentation.")
print("Run: File → Save and Export Notebook As → HTML")